# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, tell you honestly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    for attempt in range(tries):
        r = requests.get(BASE, params=params, timeout=15)

        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])

        if r.status_code not in RETRY_STATUSES:
            r.raise_for_status()

        if attempt < tries - 1:
            time.sleep(2 ** attempt)

    raise requests.HTTPError(
        f'Weather request failed after {tries} attempts',
        response=r
    )

I added a timeout and increasing retry delays for temporary network failures, HTTP 429, and server errors, while invalid requests such as 400 or 404 raise immediately.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-10**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
used_fallback = False
pull_attempted_at = pd.Timestamp.now(tz='UTC')
wx = None

try:
    weather = get_weather('2026-09-05', '2026-10-10')
    wx = weather.loc[
        weather['time'].isin(['2026-09-05', '2026-10-10'])
    ].copy()
except requests.HTTPError as error:
    if error.response.status_code not in RETRY_STATUSES:
        raise
    print(f'Weather request failed: {error}')
except (requests.Timeout, requests.ConnectionError) as error:
    print(f'Weather request failed: {error}')

I requested one date range covering both weekends and kept only the two target dates, recording any temporary request failure so the next cell can use labeled sample data.

### Part 3 — the fallback, done honestly

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-10'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if wx is None:
    used_fallback = True
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

display(wx)

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
35,2026-10-10,19.0,2.4,open-meteo


I used the provided fallback only when the request failed and labeled every row with its source so sample values cannot be mistaken for observed weather.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
35,2026-10-10,19.0,2.4,open-meteo


I renamed the weather fields and converted dates and measurements to appropriate data types so the table can be joined consistently with other datasets.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


I checked the column order, row count, datetime type, and complete, nonnegative precipitation values before using the table.

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
rain = wx.set_index('date')['precip_mm']
sept_rain = rain.loc[pd.Timestamp('2026-09-05')]
oct_rain = rain.loc[pd.Timestamp('2026-10-10')]
difference = abs(oct_rain - sept_rain)

if oct_rain > sept_rain:
    sentence = (
        f'October 10, 2026 was wetter than September 5, 2026 '
        f'by {difference:.1f} mm of precipitation.'
    )
elif sept_rain > oct_rain:
    sentence = (
        f'September 5, 2026 was wetter than October 10, 2026 '
        f'by {difference:.1f} mm of precipitation.'
    )
else:
    sentence = (
        f'Both dates received the same precipitation: '
        f'{sept_rain:.1f} mm.'
    )

label = 'Fallback sample data' if used_fallback else 'Open-Meteo data'
print(f'{label}: {sentence}')

Open-Meteo data: September 5, 2026 was wetter than October 10, 2026 by 0.1 mm of precipitation.


I compared precipitation for the two weekends and printed the wetter date and difference in millimeters, including whether the comparison uses API data or fallback samples.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print(f'Endpoint: {BASE}')
print('Coordinates: latitude 38.03, longitude -78.51')
print('Requested date range: 2026-09-05 through 2026-10-10')
print(f'Request attempted at (UTC): {pull_attempted_at}')
print(
    'Data source: fallback sample'
    if used_fallback
    else 'Data source: real weather data from Open-Meteo'
)

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Requested date range: 2026-09-05 through 2026-10-10
Request attempted at (UTC): 2026-10-12 05:45:20.408285+00:00
Data source: real weather data from Open-Meteo


I documented the endpoint, coordinates, requested dates, request timestamp, and data source so someone else can trace where the results came from.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty table instead of raising, a left merge would keep all 200 of the sale days but also give them missing weather values. A rain vs revenue graph would get rid of those rows and show no points and an inner merge would remove all the rows instead. If someone replaced missing rainfall with zero, the graph could end up misleading people and show them that every day was dry when it was just missing data.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
start = '2026-09-05'
end = '2026-10-10'
target_dates = [start, end]

orlando = get_weather(
    start=start,
    end=end,
    lat=28.54,
    lon=-81.38
)

orlando = orlando.loc[
    orlando['time'].isin(target_dates)
].copy()

orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

orlando['date'] = pd.to_datetime(orlando['date'])
orlando['temp_max'] = pd.to_numeric(orlando['temp_max'], errors='raise')
orlando['precip_mm'] = pd.to_numeric(orlando['precip_mm'], errors='raise')
orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'

charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'

weather_by_city = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)

weather_by_city = (
    weather_by_city[['city', 'date', 'temp_max', 'precip_mm', 'source']]
    .sort_values(['city', 'date'])
    .reset_index(drop=True)
)

assert len(weather_by_city) == 4, 'expected two dates per city'
assert not weather_by_city.duplicated(['city', 'date']).any()
assert weather_by_city[['temp_max', 'precip_mm']].notna().all().all()
assert (weather_by_city['precip_mm'] >= 0).all()

for city, rows in weather_by_city.groupby('city'):
    assert set(rows['date']) == set(pd.to_datetime(target_dates)), city

display(weather_by_city)

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-10,19.0,2.4,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-10,29.7,10.4,open-meteo
